# 🔢 Notebook 4: SVD — Matrix Factorization

**Mục tiêu:** Phân rã ma trận ratings → latent factors

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from surprise import SVD, KNNWithMeans, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results/charts', exist_ok=True)

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

print('✅ SVD ready!')

## 2. Train SVD

In [ ]:
model = SVD(n_factors=50, n_epochs=20, random_state=42)
trainset_full = data.build_full_trainset()
model.fit(trainset_full)
print('✅ SVD trained!')
print(f'User bias range: [{model.bu.min():.3f}, {model.bu.max():.3f}]')
print(f'Item bias range: [{model.bi.min():.3f}, {model.bi.max():.3f}]')

In [ ]:
# Vector của user 1 (dùng inner id mapping)
inner_uid = trainset_full.to_inner_uid(1)
user_vector = model.pu[inner_uid]
print(f'User 1 factor vector (5 đầu): {user_vector[:5].round(4)}')

## 3. Cross-Validation

In [ ]:
cv = cross_validate(SVD(n_factors=50, n_epochs=20, random_state=42), data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"\nCV RMSE: {cv['test_rmse'].mean():.4f} ± {cv['test_rmse'].std():.4f}")

## 4. Thử nghiệm số latent factors

In [ ]:
factors_list = [10, 20, 50, 100, 200]
rmse_list = []
for n in factors_list:
    cv = cross_validate(SVD(n_factors=n, n_epochs=20, random_state=42), data, measures=['RMSE'], cv=3, verbose=False)
    rmse_list.append(cv['test_rmse'].mean())
    print(f'  factors={n:5d}  RMSE={cv["test_rmse"].mean():.4f}')

plt.figure(figsize=(8, 4))
plt.plot(factors_list, rmse_list, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Số Latent Factors')
plt.ylabel('RMSE')
plt.title('SVD: RMSE theo Latent Factors')
plt.grid(True, alpha=0.3)
plt.savefig('results/charts/04_svd_rmse_vs_factors.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. So sánh SVD vs CF

In [ ]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)
svd_rmse = accuracy.rmse(svd.test(testset), verbose=False)

ucf = KNNWithMeans(k=40, sim_option={'name':'cosine','user_based':True}, verbose=False)
ucf.fit(trainset)
ucf_rmse = accuracy.rmse(ucf.test(testset), verbose=False)

icf = KNNWithMeans(k=40, sim_option={'name':'cosine','user_based':False}, verbose=False)
icf.fit(trainset)
icf_rmse = accuracy.rmse(icf.test(testset), verbose=False)

print(f'SVD:      RMSE = {svd_rmse:.4f}')
print(f'User-CF:  RMSE = {ucf_rmse:.4f}')
print(f'Item-CF:  RMSE = {icf_rmse:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
methods = ['User-Based CF', 'Item-Based CF', 'SVD']
vals = [ucf_rmse, icf_rmse, svd_rmse]
colors = ['steelblue', 'coral', 'seagreen']
bars = ax.bar(methods, vals, color=colors, edgecolor='black')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{v:.4f}', ha='center', fontweight='bold')
ax.set_ylabel('RMSE')
ax.set_title('So sánh RMSE: CF vs SVD')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/charts/04_algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Tổng kết

- SVD RMSE ≈ 0.87 — thường tốt hơn CF
- Nhanh hơn CF, không cần similarity matrix
- Netflix Prize winner!